# Data: how the cases are split

**Data.** PRISMS-PF dendritic solidification on the solver's adaptive (AMR) mesh, 301 frames per case (t = 0..300). Each case has three parameters: undercooling delta, anisotropy epsilonM and diffusivity D. On CHTC the frames are in `/staging/h/hsun382/me601_cache*.tar` (read with `data/amr_data.py`).

**Split.** Cases are split as whole cases (all 301 frames of a case are in the same set), so no frame of a validation or test case is ever seen in training.

| Campaign | Cases | Design | train | val | test | outer |
| --- | ---: | --- | ---: | ---: | ---: | ---: |
| `lhs100_q1` (`lhs_*`) | 100 | Latin hypercube over the training box | 80 | 10 | 10 | - |
| `lhs20_highdelta_q1` (`hd_*`) | 20 | Latin hypercube in delta 0.80-0.90 | 20 | - | - | - |
| `lhs30_fast_q1` (`fg_*`) | 30 | Latin hypercube in delta 0.72-0.90, D 1.10-1.50 | 20 | 5 | 5 | - |
| `lhs20_outer_q1` | 20 | one parameter outside the training box | - | - | - | 20 |
| total | 170 | | 120 | 15 | 15 | 20 |

Training box: delta 0.60-0.90, epsilonM 0.0375-0.0625, D 0.5-1.5.

- The high-undercooling and fast-growth cases were added where the model errors were largest (high delta, fast growth).
- The fast-growth cases are sorted by growth speed and cut into 5 groups of 6; each group gives 1 validation and 1 test case, so slow, medium and fast cases appear in every set.
- Validation is used for every model choice; test and outer are used once, for the final evaluation.

`splits.json` is the only split file; load it with `data.amr_data.load_splits`.

In [ ]:
cases_path = 'cases.csv'
split_path = 'splits.json'
out_dir = 'outputs/data'

In [ ]:
import os
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))   # P1 folder
from data.amr_data import load_splits

os.makedirs(out_dir, exist_ok=True)
cases = pd.read_csv(cases_path).set_index('case')
split = load_splits(split_path)
names = ['train', 'val', 'test', 'outer']

# Every case is in exactly one set, and splits.json agrees with cases.csv.
listed = [c for name in names for c in split[name]]
assert len(listed) == len(set(listed)) == len(cases), 'a case is missing or in two sets'
for name in names:
    assert sorted(split[name]) == sorted(cases.index[cases['set'] == name]), f'{name} differs from cases.csv'

table = pd.crosstab(cases['campaign'], cases['set'], margins=True, margins_name='total')[names + ['total']]
table

## Parameters of each set

Validation and test cases lie inside the region covered by the training cases; outer cases lie outside it on purpose.

In [ ]:
colors = {'train': '#9aa5b1', 'val': '#1f77b4', 'test': '#d62728', 'outer': '#2ca02c'}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (x, y) in zip(axes, [('delta', 'D'), ('delta', 'epsilonM')]):
    for name in names:
        c = cases[cases['set'] == name]
        ax.scatter(c[x], c[y], s=18 if name == 'train' else 40, c=colors[name], label=f'{name} ({len(c)})')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
axes[0].legend()
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'split_parameters.png'), dpi=120)